# LTV modeling - KYC + 28d

### Scope
This model is intended to offer the marketing department a robust way to evaluate the future value of acquired users shortly after they complete their KYC (28 days later). This information will allow the marketers to place reasonable amounts of money to acquiring the right users and optimising their spending according to the forecasted LTC/CAC ratio.

### Performance Criteria
The model used will have to perform adequately when forecasting the total profits expected from a cohort of users acquired in the previous month. The definition of cohorts, in other words the variables used to group users, will have to follow the marketing hierarchy framework (https://docs.google.com/spreadsheets/d/1js2bwTOt3wAfuRghV0ZxmaJSF2qi44jug9hdR62chC4/edit#gid=1909558699)

### Forecasting Horizon
After wide alignment the agreed horizon is 2 years. This number is a balance between optimising for long term profitability and setting a goal that is achievable from a modeling perspective

### Data
We use KYC data plus early activity data. All relevant DWH transformations have been aggregated in relevant dbt models


In [1]:
from datetime import datetime
import math
from os import path

from dateutil import relativedelta
from sklearn.experimental import enable_iterative_imputer
from joblib import dump, load
from sklearn.preprocessing import QuantileTransformer
from sklearn.base import clone

import sklearn
import pandas as pd
import seaborn as sns
import numpy as np
import scipy as sp
from scipy.stats import pearsonr

import matplotlib.pyplot as plt

from sklearn.preprocessing import QuantileTransformer
from sklearn.base import clone

from sklearn.metrics import (mean_squared_error as mse,
                             mean_squared_log_error as msle,
                             mean_absolute_error as mae, mean_squared_log_error as msle,
                             r2_score, roc_auc_score, roc_curve, auc, make_scorer, balanced_accuracy_score,
                             f1_score, confusion_matrix, mean_poisson_deviance, median_absolute_error, ) #mean_absolute_percentage_error

from sklearn.dummy import DummyRegressor

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import make_column_transformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, MaxAbsScaler, PowerTransformer, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer, FunctionTransformer


#from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, LassoCV, Ridge, RidgeCV, PoissonRegressor, TweedieRegressor
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor

from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedShuffleSplit, TimeSeriesSplit, GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

#from utils.helpers import get_s3_data_path, LTVForecaster

from utils.features import compute_product_features
from utils.transform import transform_user_level_data, transform_cohort_level_data, cast_dataframe, merge_tables


import pyarrow.parquet as pq
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [2]:
pip install pyarrow

You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install python-dateutil

You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
DWH_COLUMNS = {
    "user_created",
    "user_id",
    "country_tnc_legal",
    "age_group",
    "age",
    "is_expat",
    "gender",
    "newsletter_opt_in",
    "marketing_push_notifications_opt_in",
    "marketing_in_app_update_opt_in",
    "last_click_source",
    "signup_coupon",
    "first_product",
    "subscribed",
    "subscription_cost",
    "kyc_first_completed",
    "kyc_month",
    "signup_kycc_diff",
    "ftmau_flg",
    "kyc_ftmau_diff",
    "card_flg",
    "upgrade_flg",
    "kyc_card_diff",
    "cs_onboarding",
    "cs_total",
    "fees_amount",
    "amount_act_total",
    "amount_intra",
    "amount_inter",
    "amount_aa",
    "amount_ct",
    "amount_card_atm",
    "amount_grocery_market",
    "n_aa",
    "n_act_total",
    "n_intra",
    "n_inter",
    "n_card_atm",
    "n_grocery_market",
    "dau_txn",
    "dau_login",
    "dau_login_txn",
    "avg_daily_act_txns",
    "max_daily_act_txns",
    "max_diff_between_txn",
    "min_diff_between_txn",
    "avg_diff_between_txn",
    "avg_daily_logins",
    "max_daily_logins",
    "max_diff_between_logins",
    "min_diff_between_logins",
    "avg_diff_between_logins",
#    "mobile_platform",
#    "web_logins",
#    "web_logins_ratio",
    "two_y_cost",
    "two_y_revenue",
    "two_y_profit"
}

DWH_COLUMNS_COHORTS = {
    "kyc_month",
    "country_tnc_legal",
    "age_group",
    "subscribed",
    "subscription_cost",
    "is_expat",
    "gender",
    "x",
    "total_costs",
    "total_revenues",
    "total_profits",
    "avg_costs",
    "avg_revenues",
    "avg_profits",
    "std_costs",
    "std_revenues",
    "std_profits",
    "user_count",
}

COLUMNS_WITH_NAN_TO_DROP = [
    "newsletter_opt_in",
    "marketing_push_notifications_opt_in",
    "marketing_in_app_update_opt_in",
]

features = ['country_tnc_legal',
 'gender',
 'last_click_source',
 'day_name',
 'month_name',
 #'mobile_platform',
 'day_created',
 'week_of_year',
 'age',
 'fees_amount',
 'subscription_cost',
 'signup_kycc_diff',
 'kyc_ftmau_diff',
 'kyc_card_diff',
 'amount_aa',
 'amount_card_atm',
 'amount_ct',
 'amount_intra',
 'amount_inter',
 'amount_grocery_market',
 'n_aa',
 'n_grocery_market',
 'n_card_atm',
 'n_intra',
 'n_inter',
 'dau_login',
 'dau_txn',
 'cs_total',
 'avg_daily_act_txns',
 'max_daily_act_txns',
 'avg_daily_logins',
 'max_daily_logins',
 #'web_logins',
 #'web_logins_ratio',
 'max_diff_between_txn',
 'min_diff_between_txn',
 'avg_diff_between_txn',
 'max_diff_between_logins',
 'min_diff_between_logins',
 'avg_diff_between_logins',
 'avg_costs_3',
 'avg_revenues_3',
 'std_revenues_3',
 'avg_costs_2',
 'avg_revenues_2',
 'std_revenues_2',
 'user_count_2',
 'is_expat',
 'newsletter_opt_in',
 'marketing_in_app_update_opt_in',
 'signup_coupon',
 'created_weekend',
 'created_summer',
 'created_month_end',
 'upgrade_flg',
 'business_account',
 'subscribed']


blacklist = [
    "cs_onboarding",
    "ftmau_flg",
    "card_flg",
    "user_count_3",
    "month_created",
    "marketing_push_notifications_opt_in",
    "amount_act_total",
    "n_act_total",
    "std_costs_3",
    "std_costs_2",
    "first_product",
]


categorical_features = [
    "country_tnc_legal",
    "gender",
    "last_click_source",
    "day_name",
    "month_name",
    #"mobile_platform"
]


tiering = [
    "fees_amount",
    "subscription_cost",
]

acquisition = ["month_created", "day_created", "week_of_year", "age"]

contractual = [
    "signup_kycc_diff",
    "kyc_ftmau_diff",
    "kyc_card_diff",
]

user_spending = [
    "amount_act_total",
    "amount_aa",
    "amount_card_atm",
    "amount_ct",
    "amount_intra",
    "amount_inter",
    "amount_grocery_market",
    "n_aa",
    "n_grocery_market",
    "n_act_total",
    "n_card_atm",
    "n_intra",
    "n_inter",
]

user_behaviour_freq = [
    "max_diff_between_txn",
    "min_diff_between_txn",
    "avg_diff_between_txn",
    "max_diff_between_logins",
    "min_diff_between_logins",
    "avg_diff_between_logins",
]

user_behaviour = [
    "dau_login",
    "dau_txn",
    "cs_onboarding",
    "cs_total",
    "avg_daily_act_txns",
    "max_daily_act_txns",
    "avg_daily_logins",
    "max_daily_logins",
    #"web_logins",
    #"web_logins_ratio"
]



historical_behaviour = [
    "avg_costs_3",
    "avg_revenues_3",
    "std_costs_3",
    "std_revenues_3",
    "user_count_3",
    "avg_costs_2",
    "avg_revenues_2",
    "std_costs_2",
    "std_revenues_2",
    "user_count_2",
]


binary_features = [
    "is_expat",
    "newsletter_opt_in",
    "marketing_push_notifications_opt_in",
    "marketing_in_app_update_opt_in",
    "signup_coupon",
    "created_weekend",
    "created_summer",
    "created_month_end",
    "ftmau_flg",
    "card_flg",
    "upgrade_flg",
    "business_account",
    "subscribed",
    "is_top_premium",
    "is_first_premium",
]


categorical_features = [f for f in categorical_features if f not in blacklist]
acquisition = [f for f in acquisition if f not in blacklist]
contractual = [f for f in contractual if f not in blacklist]
user_spending = [f for f in user_spending if f not in blacklist]
user_behaviour = [f for f in user_behaviour if f not in blacklist]
user_behaviour_freq = [f for f in user_behaviour_freq if f not in blacklist]
historical_behaviour = [f for f in historical_behaviour if f not in blacklist]
binary_features = [f for f in binary_features if f not in blacklist]

numerical_features = (
    acquisition + tiering + contractual + user_spending + user_behaviour + user_behaviour_freq + historical_behaviour
)
features = [f for f in categorical_features + numerical_features + binary_features if f not in blacklist]



In [5]:
# read data
df_train = pd.read_csv('Data/train.csv')
df_test = pd.read_csv('Data/test.csv')

In [6]:
# reduce for local
train = df_train[:150000]
test = df_test[:100000]

In [7]:
# we keep a validation set for performing model selection later
test, validation = train_test_split(test, test_size=0.3, random_state=0)
X_train, y_train_r, y_train_c = train[features].copy(), train['two_y_revenue'], train['two_y_cost']
X_test, y_test_r, y_test_c = test[features].copy(), test['two_y_revenue'], test['two_y_cost']
X_base, y_base_r, y_base_c = test[features].copy(), test['two_y_revenue_mean'], test['two_y_cost_mean']


In [8]:
test.columns

Index(['user_created', 'user_id', 'country_tnc_legal', 'age_group', 'age',
       'is_expat', 'gender', 'newsletter_opt_in',
       'marketing_push_notifications_opt_in', 'marketing_in_app_update_opt_in',
       'last_click_source', 'signup_coupon', 'first_product', 'subscribed',
       'subscription_cost', 'kyc_first_completed', 'kyc_month',
       'signup_kycc_diff', 'active_months', 'first_active', 'ftmau_flg',
       'kyc_ftmau_diff', 'card_flg', 'upgrade_flg', 'kyc_card_diff',
       'cs_onboarding', 'cs_total', 'fees_amount', 'amount_act_total',
       'amount_intra', 'amount_inter', 'amount_aa', 'amount_ct',
       'amount_card_atm', 'amount_grocery_market', 'n_aa', 'n_act_total',
       'n_intra', 'n_inter', 'n_card_atm', 'n_grocery_market', 'dau_txn',
       'dau_login', 'dau_login_txn', 'avg_daily_act_txns',
       'max_daily_act_txns', 'max_diff_between_txn', 'min_diff_between_txn',
       'avg_diff_between_txn', 'avg_daily_logins', 'max_daily_logins',
       'max_diff_betwe

### Training & Model Selection


In [9]:
def _revenue_weight(bucket):
    if bucket == 1:
        return 3
    if bucket == 2:
        return 2
    return 1


def _cost_weight(bucket):
    if bucket == 1:
        return 2
    return 1


revenue_bin = KBinsDiscretizer(20, strategy='quantile', encode='ordinal').fit_transform(y_train_r.values.reshape(-1, 1)) + 1
cost_bin = KBinsDiscretizer(20, strategy='quantile', encode='ordinal').fit_transform(y_train_c.values.reshape(-1, 1)) + 1

revenue_weight = np.apply_along_axis(_revenue_weight, 1, arr=revenue_bin) + train.country_tnc_legal.isin(['ITA']) * 2 + train.first_product.isin(["BUSINESS_CARD"])
cost_weight = np.apply_along_axis(_cost_weight, 1, arr=cost_bin) + train.country_tnc_legal.isin(['AUT', 'DEU', 'ITA', 'ESP'])

# preprocess data
_blacklist = [
    'std_revenues_3',
    'amount_intra',
    'n_intra',
    'created_month_end',
    'month_name',
    'subscribed',
    'created_summer',
]

# the features in the _blacklist are not processed by the column transformer and dropped from train/inference. Nevertheless those variables
# may still be useful for EDA therefore we don't remove them completely from the repo
preprocessor = make_column_transformer(
    (
        StandardScaler(), [f for f in acquisition if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder()), [f for f in categorical_features if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='constant', fill_value=90), RobustScaler()), [f for f in contractual if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), [f for f in tiering if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='constant', fill_value=0), RobustScaler()), [f for f in user_spending if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='median'), RobustScaler()), [f for f in historical_behaviour if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='constant', fill_value=0), RobustScaler()), [f for f in user_behaviour if f not in _blacklist]
    ),
    (
        make_pipeline(SimpleImputer(strategy='median', fill_value=60), RobustScaler()), [f for f in user_behaviour_freq if f not in _blacklist]
    ),
    (
        SimpleImputer(strategy='constant', fill_value=0), [f for f in binary_features if f not in _blacklist]
    ),
    remainder='drop'
)


/usr/local/lib/python3.7/site-packages/sklearn/preprocessing/_discretization.py:197: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  'decreasing the number of bins.' % jj)
/usr/local/lib/python3.7/site-packages/sklearn/preprocessing/_discretization.py:197: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  'decreasing the number of bins.' % jj)


In [18]:
    # run model
    global_revenue_model = make_pipeline(
        clone(preprocessor),
        HistGradientBoostingRegressor(),
    ).fit(
        X_train, y_train_r,
        #histgradientboostingregressor__sample_weight=revenue_weight
    )
    pred_r = global_revenue_model.predict(X_train)

In [19]:
    global_cost_model = make_pipeline(
        clone(preprocessor),
        HistGradientBoostingRegressor(),
    ).fit(
        X_train, y_train_c,
        #histgradientboostingregressor__sample_weight=cost_weight
    )
    pred_c = global_cost_model.predict(X_train)

    

In [20]:
test.columns
#test['revenue_forecast'] = pd.DataFrame(pred_c)
#test['cost_forecast'] = pd.DataFrame(data =pred_c)

Index(['user_created', 'user_id', 'country_tnc_legal', 'age_group', 'age',
       'is_expat', 'gender', 'newsletter_opt_in',
       'marketing_push_notifications_opt_in', 'marketing_in_app_update_opt_in',
       'last_click_source', 'signup_coupon', 'first_product', 'subscribed',
       'subscription_cost', 'kyc_first_completed', 'kyc_month',
       'signup_kycc_diff', 'active_months', 'first_active', 'ftmau_flg',
       'kyc_ftmau_diff', 'card_flg', 'upgrade_flg', 'kyc_card_diff',
       'cs_onboarding', 'cs_total', 'fees_amount', 'amount_act_total',
       'amount_intra', 'amount_inter', 'amount_aa', 'amount_ct',
       'amount_card_atm', 'amount_grocery_market', 'n_aa', 'n_act_total',
       'n_intra', 'n_inter', 'n_card_atm', 'n_grocery_market', 'dau_txn',
       'dau_login', 'dau_login_txn', 'avg_daily_act_txns',
       'max_daily_act_txns', 'max_diff_between_txn', 'min_diff_between_txn',
       'avg_diff_between_txn', 'avg_daily_logins', 'max_daily_logins',
       'max_diff_betwe

In [10]:
#test = test[~test.two_y_profit.isna()]

profit, revenue, cost = True, False, False

if profit is True:
    pred, actual, baseline = 'profit_forecast', 'two_y_profit', 'two_y_profit_mean'
elif revenue is True:
    pred, actual, baseline = 'revenue_forecast', 'two_y_revenue', 'two_y_revenue_mean'
elif cost is True:
    pred, actual, baseline = 'cost_forecast', 'two_y_cost', 'two_y_cost_mean' 

for dataset_name, dataset, predictor in [
    ('TRAIN', train, pred), ('TEST', test, pred), ('TEST', test, baseline)
]:
    print('[{} - model {}]'.format(dataset_name, predictor))
    for metric_name, metric in [
        ('R2 score', r2_score), ('Mean Absolute Error', mae), ('Median Absolute Error', median_absolute_error), ('Mean Squared Error', mse), 
    ]:
        print('{} - {}'.format(
            metric_name, round(metric(dataset[actual], dataset[predictor]), 4)
        ))
    print('')

[TRAIN - model profit_forecast]
R2 score - 0.2618
Mean Absolute Error - 47.0879
Median Absolute Error - 22.2226
Mean Squared Error - 7535.5034

[TEST - model profit_forecast]
R2 score - 0.334
Mean Absolute Error - 39.4962
Median Absolute Error - 15.3851
Mean Squared Error - 5826.2248

[TEST - model two_y_profit_mean]
R2 score - 0.1966
Mean Absolute Error - 44.4065
Median Absolute Error - 19.0727
Mean Squared Error - 7028.439



- [validation month 2018-10-01] performance of ml forecasting model 12.43%, performance of cohorts mean model 29.77%
- [validation month 2018-11-01] performance of ml forecasting model 21.39%, performance of cohorts mean model 37.29%
- [validation month 2018-12-01] performance of ml forecasting model 26.75%, performance of cohorts mean model 37.2%
- [validation month 2019-01-01] performance of ml forecasting model 33.15%, performance of cohorts mean model 28.78%
- Avg ML model weighted percentage error 23%
- Avg mean model weighted percentage error 33%

In [169]:
tt = []
model_perf = []
mean_model_perf = []
t = test[test.kyc_month == pd.to_datetime('2018-10-01').strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product'])[
        ['two_y_profit', 'profit_forecast', 'two_y_profit_mean']
    ].sum()
t['two_y_profit'] = t['two_y_profit'].apply(lambda p: 1 if p == 0 else p)


t['pe_forecast'] = (np.abs(t['two_y_profit'] - t['profit_forecast']) / np.abs(t['two_y_profit'])) * 100
t['pe_mean'] = (np.abs(t['two_y_profit'] - t['two_y_profit_mean']) / np.abs(t['two_y_profit'])) * 100


t = t.merge(
    test[test.kyc_month == pd.to_datetime(month).strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product']).user_created.count().rename('user_count'),
    on=group
)

t['weight'] = t['two_y_profit'].abs() / t['two_y_profit'].sum()

t['pe_forecast_w'] = t['pe_forecast'] * t['weight']
t['pe_mean_w'] = t['pe_mean'] * t['weight']
print('[validation month {}] performance of ml forecasting model {}%, performance of cohorts mean model {}%'.format(
    month, t['pe_forecast_w'].sum().round(2), t['pe_mean_w'].sum().round(2))
)
tt.append(t)
model_perf.append(t['pe_forecast_w'].sum())
mean_model_perf.append(t['pe_mean_w'].sum())

[validation month 2019-02-01] performance of ml forecasting model 12.43%, performance of cohorts mean model 29.77%


In [168]:
#2018-11-01
tt = []
model_perf = []
mean_model_perf = []
t = test[test.kyc_month == pd.to_datetime('2018-11-01').strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product'])[
        ['two_y_profit', 'profit_forecast', 'two_y_profit_mean']
    ].sum()
t['two_y_profit'] = t['two_y_profit'].apply(lambda p: 1 if p == 0 else p)


t['pe_forecast'] = (np.abs(t['two_y_profit'] - t['profit_forecast']) / np.abs(t['two_y_profit'])) * 100
t['pe_mean'] = (np.abs(t['two_y_profit'] - t['two_y_profit_mean']) / np.abs(t['two_y_profit'])) * 100


t = t.merge(
    test[test.kyc_month == pd.to_datetime(month).strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product']).user_created.count().rename('user_count'),
    on=group
)

t['weight'] = t['two_y_profit'].abs() / t['two_y_profit'].sum()

t['pe_forecast_w'] = t['pe_forecast'] * t['weight']
t['pe_mean_w'] = t['pe_mean'] * t['weight']
print('[validation month {}] performance of ml forecasting model {}%, performance of cohorts mean model {}%'.format(
    month, t['pe_forecast_w'].sum().round(2), t['pe_mean_w'].sum().round(2))
)
tt.append(t)
model_perf.append(t['pe_forecast_w'].sum())
mean_model_perf.append(t['pe_mean_w'].sum())

[validation month 2019-02-01] performance of ml forecasting model 21.39%, performance of cohorts mean model 37.29%


In [167]:
tt = []
model_perf = []
mean_model_perf = []
t = test[test.kyc_month == pd.to_datetime('2018-12-01').strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product'])[
        ['two_y_profit', 'profit_forecast', 'two_y_profit_mean']
    ].sum()
t['two_y_profit'] = t['two_y_profit'].apply(lambda p: 1 if p == 0 else p)


t['pe_forecast'] = (np.abs(t['two_y_profit'] - t['profit_forecast']) / np.abs(t['two_y_profit'])) * 100
t['pe_mean'] = (np.abs(t['two_y_profit'] - t['two_y_profit_mean']) / np.abs(t['two_y_profit'])) * 100


t = t.merge(
    test[test.kyc_month == pd.to_datetime(month).strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product']).user_created.count().rename('user_count'),
    on=group
)

t['weight'] = t['two_y_profit'].abs() / t['two_y_profit'].sum()

t['pe_forecast_w'] = t['pe_forecast'] * t['weight']
t['pe_mean_w'] = t['pe_mean'] * t['weight']
print('[validation month {}] performance of ml forecasting model {}%, performance of cohorts mean model {}%'.format(
    month, t['pe_forecast_w'].sum().round(2), t['pe_mean_w'].sum().round(2))
)
tt.append(t)
model_perf.append(t['pe_forecast_w'].sum())
mean_model_perf.append(t['pe_mean_w'].sum())

[validation month 2019-02-01] performance of ml forecasting model 26.75%, performance of cohorts mean model 37.2%


In [166]:
tt = []
model_perf = []
mean_model_perf = []
t = test[test.kyc_month == pd.to_datetime('2019-01-01').strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product'])[
        ['two_y_profit', 'profit_forecast', 'two_y_profit_mean']
    ].sum()
t['two_y_profit'] = t['two_y_profit'].apply(lambda p: 1 if p == 0 else p)


t['pe_forecast'] = (np.abs(t['two_y_profit'] - t['profit_forecast']) / np.abs(t['two_y_profit'])) * 100
t['pe_mean'] = (np.abs(t['two_y_profit'] - t['two_y_profit_mean']) / np.abs(t['two_y_profit'])) * 100


t = t.merge(
    test[test.kyc_month == pd.to_datetime(month).strftime('%Y-%m-%d')].groupby(['country_tnc_legal', 'first_product']).user_created.count().rename('user_count'),
    on=group
)

t['weight'] = t['two_y_profit'].abs() / t['two_y_profit'].sum()

t['pe_forecast_w'] = t['pe_forecast'] * t['weight']
t['pe_mean_w'] = t['pe_mean'] * t['weight']
print('[validation month {}] performance of ml forecasting model {}%, performance of cohorts mean model {}%'.format(
    month, t['pe_forecast_w'].sum().round(2), t['pe_mean_w'].sum().round(2))
)
tt.append(t)
model_perf.append(t['pe_forecast_w'].sum())
mean_model_perf.append(t['pe_mean_w'].sum())

[validation month 2019-02-01] performance of ml forecasting model 33.15%, performance of cohorts mean model 28.78%


In [77]:
# X_train.to_csv('X_train.csv', index=False)
y_train_c.to_frame().to_csv('y_train_c.csv', index=False)
y_train_r.to_frame().to_csv('y_train_r.csv', index=False)
train[['revenue_weight', 'cost_weight']].to_csv('training_weights_ikaria.csv', index=False)
revenue_features = permutation_importance(
    poisson_r, X_test, y_test_r,
    n_repeats=30, random_state=0, n_jobs=-1
)
%matplotlib widget


importances = []
names = []
for i in revenue_features.importances_mean.argsort()[::-1]:
    if revenue_features.importances_mean[i] - 2 * revenue_features.importances_std[i] > 0:
        importances.append(revenue_features.importances_mean[i])
        names.append(features[i])

g = sns.barplot(y=names, x=importances)
cost_features = permutation_importance(
    poisson_c, X_test, y_test_c,
    n_repeats=30, random_state=0, n_jobs=-1
)
%matplotlib widget


importances = []
names = []
for i in cost_features.importances_mean.argsort()[::-1]:
    if cost_features.importances_mean[i] - 2 * cost_features.importances_std[i] > 0:
        importances.append(cost_features.importances_mean[i])
        names.append(features[i])

g = sns.barplot(y=names, x=importances)

NameError: name 'poisson_r' is not defined

In [ ]:
# for s3
os.environ['LOCAL_USER'] = "ramsey.wise"
config_file = "/app/utils/CONFIG.ini"

In [ ]:
# query redshift
def df_from_db(query, db):
    with db.engine.begin() as conn:
        return db.df_from_sql(query, connection=conn)

db_redshift = connect_redshift(config_file, "local")

In [ ]:
# read s3
s3_data_path_users = get_s3_data_path(data_type="users", env="local", date_min="2017-06-01", date_max="2019-06-01", s3_bucket=s3_bucket)
s3_data_path_cohorts = get_s3_data_path(data_type="cohorts", env="local", date_min="2017-01-01", date_max="2020-01-01", s3_bucket=s3_bucket)
print(f"S3 path {s3_data_path_users}")

unload_user_data_from_dwh.run(
    db=db_redshift, min_signup_date="2017-01-01", min_kyc_date="2017-06-01", max_kyc_date="2019-06-30",
    redshift_iam_arn=redshift_iam_arn, s3_data_path_users=path.join(s3_data_path_users, "users")
)

unload_cohort_data_from_dwh.run(
    db=db_redshift, min_kyc_month="2017-01-01", max_kyc_month="2020-01-01",
    redshift_iam_arn=redshift_iam_arn, s3_data_path_cohorts=path.join(s3_data_path_cohorts, "cohorts")
)

df = read_data_from_s3.run(s3_data_path_users)
df_cohorts = read_data_from_s3.run(s3_data_path_cohorts)
df = cast_dataframe.run(df)
df_cohorts = cast_dataframe.run(df_cohorts)

perform_data_quality_checks.run(
    df, "2017-06-01", "2019-06-30",
    set(list(DWH_COLUMNS) + ["two_y_cost", "two_y_revenue", "two_y_profit"]),
    ACCEPTED_MISSING_RATIO
)

perform_data_quality_checks.run(
    df_cohorts, "2017-01-01", "2020-01-01",
    DWH_COLUMNS_COHORTS, ACCEPTED_MISSING_RATIO_COHORTS
)
df = transform_user_level_data.run(
    df, COLUMNS_WITH_NAN_TO_DROP, ACCEPTED_DROP_RATIO
)

df_cohorts = transform_cohort_level_data.run(
    df_cohorts, COLUMNS_WITH_NAN_TO_DROP_COHORTS, ACCEPTED_DROP_RATIO_COHORTS
)

In [ ]:
# Removing users with outliying values in the most important numerical features
outlier_features = ["amount_ct", "avg_daily_act_txns", "amount_aa", "amount_card_atm", "n_aa", "max_daily_logins", "avg_revenues_3", "avg_costs_3", "avg_revenues_2"]

preprocessor_outlier = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), outlier_features),
)

outliers = IsolationForest(
    random_state=0, contamination=0.005, n_jobs=-1,
).fit_predict(preprocessor_outlier.fit_transform(train[outlier_features]))

non_outliers_msk = outliers == 1

print((outliers == -1).astype(int).sum())
# ltv_model = load("ltv_forecaster_model_2021-05-28_v1.1.joblib")
train_subscribed, train_non_subscribed, train_global = True, True, True
